# Interpretable spatial domains with Kontext. 

*Kontext is a project developped by Anthony Ozier-Lafontaine, Paul Villoutreix and Laura Cantini, 
in the Machine Learning for Integrative Genomics (ML4IG) team of Pasteur institute led by Laura Cantini. The associated preprint and the python package will be available at the end of september 2026.*

## Introduction 

Kontext finds spatial domains using the statistical framework of kernel methods. 
It is performant, fast, has low memory usage and is very interpretable. 
Spatial domains can be interpreted in terms of cell-cell communication, niche analysis, local alignment, and many others.  
Kontext is a python package entirely compatible with the scverse ecosystem. It relies on standard machine learning packages such as numpy, pandas, scipy and scikit-learn. 


## Goal of the workshop 

The goal of this notebook is to introduce you to the Kontext framework for your spatial transcriptomics analyses. 


## Kontext in a nutshell

### Spatial domains 

Given a spatial transcriptomics dataset of $n$ observations and $G$ genes, Kontext relies on the count matrix $X$ of size $n \times G$ and the local microenvironment matrix $C$ of size $n \times G$. 
The value $C_{i,g}$ summarizes the presence of gene $g$ in the local neighborhood of cell $i$. 
Each neighbor of cell $i$ is given a weight $w_{ij}$ built on spatual proximity $k_s$ and expression similarity $k_x$. 
Both kernel functions $k_s$ and $k_x$ are associated to one parameter each, a number of neighbors `snn` and a bandwidth ratio `sigma_e`respectively.

The spatial domain embedding of cell $i$ is a convex combination of $X_i$ and $C_i$, of balance parameter $\alpha$ : 

$$E_i = \alpha X_i + (1-\alpha)C_i$$

### Downstream tasks

The expression matrix $X$, weight matrix $W$, and microenvironment matrix $C$ can be leveraged for various analyses, use to interpret Kontext's spatial domains. 

- **Cell-cell communication** : A ligand $L$ and receptor $R$ are probably interacting if the receptor transcript is found in the cell ($X_{i,R}>0$) and the ligand transcript is expressed in the cells nearby $(C_{i,L}>0)$. The Kontext communication score is thus :
$$S_{i,(L,R)} = X_{i,R} \times C_{i,L}. $$
- **Sender-receiver communication** : Consider three cell types $a,b$ and $c$, each cell type contribution to the local microenvironment can be quantified such that $C_i = C_i^a + C_i^b + C_i^c$. Based on this, given a cell $i$ in a receiver group, the sender-receiver communication score of cell $i$ with respect to sender group $a$ is : $$S_{i,(L,R)}^a = X_{i,R} + C_{i,L}^a$$
- **Niche analysis** : THe weight matrix $W$ can also be used to quantify cell type presence in the local microenvironment. This can be used for co-localization analysis or boundary detection. 
$$N_i = \sum_j w_{i,j} group(j)$$
- **Local alignment** : How much a cell is aligned with its local microenvironment ? The local alignment scores quantifies this.  $$LA_i = similarity(X_i,C_i)$$


## Presentation of the dataset

We analyse a Xenium dataset containing 3 time points (sham (2 sections), 7 days post injury (3 sections), 28 days post injury (3 sections)) from mouse heart recovering from an injury. (https://www.nature.com/articles/s44161-025-00739-6)

# 1 - Load data 

In [ ]:
%load_ext autoreload
%autoreload 2
import warnings; warnings.simplefilter('ignore', FutureWarning)
import scanpy as sc; sc.settings.verbosity = 1

In [ ]:

adata = sc.read_h5ad('../data/cryoinjury.h5ad')
print(adata)

# Access to cell information 
# print(adata.obs.columns)

# Access to dataset size 
# print(adata.shape)

# Access to the gene panel 
# print(adata.var_names)

As you can see, this Xenium dataset contains 262133 cells and 540 genes. 

Cell information are contained in dataframe `adata.obs`. 
Column `cell_type` contains cell type annotations from the original publication, summarized into main groups in column `main_cell_type`. 

Column `time` assings each cell to its time point :
- `sham` : sham (2 samples) 
- `cryo7D` : 7 days post injury (3 samples) 
- `cryo28D` : 28 days post injury (3 samples)

Column `sample` assigns each cell to its sample of origin :
- sham : `sham_s1`, `sham_s2`
- 7 days post injury : `cryo7D_S1`, `cryo7D_S2`, `cryo7D_S2`
- 28 days post injury : `cryo28D_S1`, `cryo28D_S2`, `cryo28D_S2`


**Question.** 

Split the dataset per sample to plot their spatial organisation. 

In [ ]:
import matplotlib.pyplot as plt
from kontext.visualization import get_figs,space_plot

samples = ['sham_S1','sham_S2',
            'cryo7D_S1','cryo7D_S2','cryo7D_S3',        
            'cryo28D_S1','cryo28D_S2','cryo28D_S3',]

for sample in samples : 
    adata_sample = adata[adata.obs['sample']==sample].copy()

    fig,ax = get_figs(1) # Helper function to initiate an empty figure (1 corresponds to the number of plots within the figure).
    space_plot(adata_sample,'main_cell_type',ax=ax,s=10) # Helper function for space plotting. `s`` is the size of the dots. 
    ax.set_title(sample) 
    plt.show()


# 2 - Kontext spatial domains 

Few steps are necessary to obtain spatial domains with Kontext. 

### Parameters definition 

We need to fix three groups of parameters : 
- `preprocessing`: How to preprocess the data with respect to the technology. 
- `embedding` : Parameters of spatial proximity, expression similarity and convex combination balance parameter. 
- `spatial_domains` : Clustering algorithm and its parameters (random seed, number of clusters) 

*All groups are stored in dictionaries*

### Preprocessing

Kontext is efficient and scalable because it keeps the data sparse throughout the whole pipeline. 
We propose technology specific preprocessing. All have `log1p` ($X = log(X+1)$) and `L2-normalization` ($\Vert x_i\Vert^2=1$) as final steps. 
Other steps can be :
- **Gene filtering** : remove genes present in less than 20 cells. 
- **HVG selection** : remove genes not present in the 2000 most highly variable genes. 
- **LR** : Reintroduce genes involved in known ligand-receptor pairs if they were discarded during **gene filtering** or **HVG selection**. 

For full transcriptome sequencing-based technologies : 
- `sparse` : Gene filtering +  `log1p` + `L2-normalization`
- `sparse_hvg` : Gene filtering + HVG selection + `log1p` + `L2-normalization`
- `spase_hvg_LR` : Gene filtering + HVG selection + LR + `log1p` + `L2-normalization`

For imaging-based technologies such as MERFISH or Xenium : 
- `image` : `log1p` + `L2-normalization`

**Question.**

We are working with a Xenium dataset, choose the right preprocessing option and preprocess the dataset. 

In [ ]:
from kontext.data import preprocess_adata

preprocessing = {'preprocessing': # to fill 
                 }
adata = preprocess_adata(adata,**preprocessing)  

### Embedding 

Parameters of spatial proximity, expression similarity and convex combination balance parameter. 
To compute weights $W$ and microenvironment embeddings $C$
- `sigma_e` ($\geq 0$, default: 1): parameter of the expression similarity kernel. 
- `snn` (integer, default: 20) : number of neighbors to consider around each cell. 

To compute spatial domains embeddings $E$. 
- `alpha`(in $[0,1]$, default: 0.5) : balance between expression $X$ and local microenvironment $C$. Choose $0$ for microenvironment only and $1$ for expression only. 

**Question.** 

In our analysis, we used the default. Choose your parameters and initiate your Kontext object. 

In [ ]:
from kontext.kontext_class import Kontext


embedding = {"sigma_e":  # to fill,  
             "snn": # to fill, 
             "alpha": # to fill, 
             }


# The Kontext object does not contains any data, its purpose it to interact and update the AnnData object.
ko = Kontext(**embedding,verbose=False)

Actually, to ensure that spatial positions from different sections will not interfere with each other, we split the dataset to compute weights on each section separately then we merge them.  

- `ko.weights()` : computes the weight matrix `wc` (spatial proximity `ws`, expression similarity `we` and a trick weight matrix used to compute $E$ directly `w` are also kept in memory). 

In [ ]:
from kontext.data import merge_adatas


adatas = {}  # used to reconstruct the full AnnData object 
for sample in samples:
    adata_sample = adata[adata.obs['sample']==sample].copy() # subset the cells from 1 sample 
    adata_sample.uns = adata.uns.copy() # ensure separate instances 
    adata_sample.uns['data']['sample']=sample # store sample information 

    ko.weights(adata_sample)

    adatas[sample] = adata_sample 

adata = merge_adatas(adatas,block_diag_keys=['w','ws','wc','we'])
del adatas     

This cell is here to show you how to compute and access the microenvironment matrix $C$ and spatial domains matrix $E$. 
It is actually unnecessary as functions who need these matrices will compute them automatically if they are not available. 

- `ko.microenvironment_embedding`: computes $C$.
- `ko.spatial_domains_embedding`: computes $E$. 

*$C$ and $E$ have the same dimension than $X$, thus they are stored in `adata.layers`.*

In [ ]:
ko.microenvironment_embedding(adata)
ko.spatial_domains_embedding(adata)

print(adata)
print('C : ',adata.layers['C'].shape)
print('E : ',adata.layers['E'].shape)

Looking at the umap plot of an embedding is always insightful. Here is how to do it. It may take more time than other functions (few minutes). 

**Question (if you have time).** 

Choose a number of principal components `n_comps` (default: 50), the `layer` of which you want to see the umap ($X$, $C$ or $E$) and an information from `adata.obs` to display as the color. Then compute the PCA and UMAP. 

In [ ]:
from umap.umap_ import UMAP 
from kontext.visualization import umap_plot
from kontext.utils import pca_of_mat 

n_comps = # to fill 
color =  # choose any column present in adata.obs (e.g. time, sample, cell_type, main_cell_type)

### Uncomment your choice : 

### PCA of the count matrix X 
# layer = 'X'
# pca_of_mat(adata,adata.X,n_comps=n_comps,key=layer)

### PCA of the microenvironment embedding 
# layer = 'C'
# pca_of_mat(adata,adata.layers[layer],n_comps = n_comps)

### PCA of the microenvironment embedding 
# layer = 'E'
# pca_of_mat(adata,adata.layers[layer],n_comps = n_comps)


# UMAP 
adata.obsm[f'{layer}_umap'] = UMAP(n_components=2).fit_transform(adata.obsm[f'{layer}_pca'])


fig,ax = get_figs(1)
umap_plot(adata,color,ax=ax,umap_key=f'{layer}_umap') # same structure than space_plot


### Spatial domains 

Everything is ready to compute the spatial domains.  
We currently propose the Gaussian mixture model algorithm `gmm` and the Kmeans clustering `kmeans` (Leiden and Louvain will to be added later). 

- `ko.spatial_domains()` : compute spatial domains and stores them in `adata.obs['spatial_domains']`.

**Question. (no wrong answer)**

Choose a number of domains `n_domains`, a clustering algorithm `clustering` and a random seed `seed` and compute spatial domains with `ko.spatial_domains()`.
(Our analysis was done with $8$ spatial domains, gmm clustering and seed 0). 

In [ ]:

n_domains =  # to fill 
seed = # to fill 
spatial_domains = {'n_domains':n_domains,'clustering':'gmm','seed':seed}
ko.spatial_domains(adata,**spatial_domains)

print(adata.obs['spatial_domains'])


We introduce an helper function 

- `space_plot_mask` : Spatial plot where the only colored cells come from a specific subpopulation (e.g. a spatial domain). The focus is on cells corresponding to value `group` in `adata.obs` column `col`. 

**Question.** 

Select a sample to display and space plot the spatial organization of your domains. 

*Note that you do not need to reconstruct the complete adata object after that as no information were added to the subsamples.* 

In [ ]:
from kontext.visualization import space_plot
### This cell is used to overcome a technical artifact in scanpy 
### that needs to plot the domains to have the same colors across all samples. 
### The resulting plot shows all the samples superposed and is not readable. 
space_plot(adata,'spatial_domains')

### Alternatively, you can simply hard code the spatial domain colors (here for 8 spatial domains): 
# spatial_domains_colors = [ 
#     "#0072B2","#D55E00","#009E73","#CC79A7",
#     "#E69F00","#56B4E9","#F0E442","#4E1A1A",]
# adata.uns['spatial_domains_colors'] = spatial_domains_colors


# Same artifact for the cell types, we hardcode the main_cell_type colors. 
cell_type_colors = [
    "#8DD3C7","#FFFFB3","#BEBADA","#FB8072",
    "#80B1D3","#FDB462","#B3DE69","#FCCDE5",
    "#D9D9D9","#BC80BD","#CCEBC5", "#FFED6F",
    "#DCD685","#D7AFAF",]
adata.uns['main_cell_type_colors'] = cell_type_colors

In [ ]:
from kontext.visualization import space_plot_mask

samples = ['sham_S1','sham_S2',
            'cryo7D_S1','cryo7D_S2','cryo7D_S3',        
            'cryo28D_S1','cryo28D_S2','cryo28D_S3',]

sample= # to fill

adata_sample = adata[adata.obs['sample']==sample].copy()

fig,axes = get_figs(n_domains,4,5)
for ax,domain in zip(axes,range(n_domains)):
    mask = space_plot_mask(adata_sample,
                           col='spatial_domains',
                           group=domain,
                            # color = 'main_cell_type',   # you should try this 
                           ax=ax,snm=5,sm=50)


Domains spatial positions is great, but cell type compositions is also informative. 

**Question.**

Use the function `get_proportions()` to obtain a table that displays the proportions of `col2` (e.g. `main_cell_type`) in each category of `col1` (e.g. `spatial_domains`). 

Then use the function `plot_region_distribution()` to display this table in a readable way. 

In [ ]:
from kontext.visualization import get_proportions, plot_region_distribution

col1 = # to fill 
col2 = # to fill 

proportions = get_proportions(adata,col1,col2,normalize=False)
print(proportions)

fig,ax = get_figs(1)
plot_region_distribution(proportions,colors = adata.uns[f'{col2}_colors'] ,ax=ax,fontsize=12,format_value='.2f',threshold=.1,percentage=True)


# 3 - Downstream tasks 

## 3.1 - Cell-cell communication 

We now interpret spatial domains in terms of ligand-receptor interactions. 

- `get_LR_pairs()` : identifies the ligand-receptor pairs present in the dataset for this organism (`mouse`) using the Liana database (https://www.nature.com/articles/s41556-024-01469-w).  

**Question.** 

Explore the LR pairs available in this Xenium panel, you might know some of them. 

In [ ]:
from kontext.utils import get_LR_pairs

pairs = get_LR_pairs(adata, organism='mouse', verbose=False)
print(pairs)

For our analysis, we focused on some specific pairs of interest. 

In [ ]:
pairs_of_interest = [
    'Cfh_Itgam','C3_Itgam','C3_C3ar1',
    'Ccl12_Ccr2','Spp1_Cd44','Mmp9_Cd44','Ptn_Sdc1',
    'Thbs4_Cd36','Thbs4_Sdc1','Comp_Sdc1',
    'Bmp7_Eng',
    'Timp3_Cd44','S100a4_Erbb2',
    'Timp3_Kdr','Angpt2_Tie1',]

Now we compute the communication scores 

- `ko.cell_cell_communication()` : computes the LR scores and averages across all LR pairs to derive cell communication scores stored in `adata.obs['cell_communication']`. 
- `plot_ccc_score_per_group()` : helper function to visually the average cell communication scores per category of `group_col` (e.g. `spatial_domains`). You can display the region associated to each bar in a specific sample with parameters (`column_shown`,`sample_shown`).

**Question.**

Compute and plot the cell communication scores. 

In [ ]:
from kontext.visualization import plot_ccc_score_per_group

fig,ax = get_figs(1)
ko.cell_cell_communication(adata,pairs,mode='mean')

plot_ccc_score_per_group(adata,
                         group_col = 'spatial_domains',
                         ccc_score_col = 'cell_communication',
                         column_shown ='sample',
                         sample_shown='cryo7D_S1', 
                         ax=ax, insersion=(.10, .12))

### 3.2 - Sender-receiver communication 

Lets focus on sender-receiver communication scores between two cell populations. 

**Question :**

Choose two cell populations `t1`and `t2` from the same column `groups`and display their communication scores over time then over spatial domains. (In our work, we focused on the communication of `main_cell_types` `Fibroblast` and `Macrophage` over time.)

In [ ]:
import seaborn as sns

groups = # to fill
t1, t2 = # to fill 
times = 'sham','cryo7D','cryo28D'


for receiver,sender in (t1, t2),(t2,t1),(t1, t1), (t2, t2): 
    ko.init_sender_receiver_communication(adata,pairs,sender,receiver,groups)

for receiver,sender in (t1, t2),(t2,t1),(t1, t1), (t2, t2): 
    T = ko.sender_receiver_scores(adata,pairs,sender,receiver,groups,receiver_threshold=0.1,
                                  group_by='time',
                                  by_order=times # to use if you want a given order in the resulting table, None otherwise.  
                                  )
    # T = T[[p for p in pairs_of_interest if p in T.columns]]   # to focus on a list of pairs of interest

    fig,ax = get_figs(1)
    hm = sns.heatmap(T.T, ax=ax, cmap='RdBu_r', center=0, vmin=-6, vmax=4,
                annot=True, fmt='.2f', annot_kws={'fontsize':15},
                linewidths=.5,) 
    ax.set_title(f'{sender} to {receiver}')

The limitation of this analysis is that the compared groups should be encoded within the same `adata.obs` column. 
To overcome this limitation and apply it on custom groups (e.g. macrophages in spatial domain 1 with fibroblasts in spatial domain 1 over time), one has to create a custom column in `adata.obs`. 

**Question (not mandatory)** 

Choose two specific groups you want to compare, construct the custom column and display the resulting scores.  

In [ ]:
import pandas as pd 

# example of a custom column 
adata.obs[f'type_domain'] = pd.Categorical(adata.obs['main_cell_type'].astype(str) + '_' + adata.obs[f'spatial_domains'].astype(str))

# To fill 
# 
# 

### 3.3 Niche analysis. 

- `ko.niche()`: Given an `adata.obs` column  `cell_type_key` (e.g. `spatial_domains`), the niche vector of a cell quantifies the presence of each category of the column in its local microenvironment. Each niche value is stored in `adata.obs[f'niche_{cell_type_key}_{category}']` and the niche matrix is in `adata.obsm[f'niche_{cell_type_key}']`

**Question.** 

Compute a niche matrix and explore it. 

In [ ]:
cell_type_key = 'spatial_domains' # to fill
ko.niche(adata, cell_type_key=cell_type_key)



category = # to fill
print(adata.obs[f'niche_{cell_type_key}_{category}'].sum() )
print(sample)

adata.obs['sample']= adata.obs['sample'].astype('category')
fig,axes = get_figs(len(samples))
for sample,ax in zip(samples,axes) :
    adata_sample = adata[adata.obs['sample']==sample]
    # space_plot(adata_sample,f'niche_{cell_type_key}_{category}',ax=ax)


We can perform a niche enrichment analysis 

In [ ]:
from kontext.utils import niche_enrichment
enriched = niche_enrichment(adata,'main_cell_type',['Fibroblast','Macrophage'],key = 'niche_spatial_domains')
print(enriched)

### 3.4 Local alignment 

The local alignment of a cell with its microenvironment is computed with cosine similarity between $X$ and $C$ through `ko.local_alignment()` and stored in `adata.obs['local_alignment']`

**Question**

Compute and display the local alignment of your dataset. 

In [ ]:
ko.local_alignment(adata)

# to fill 

## 4  Just for fun : Become a computational biologist yourself.  

There are plenty of other analyses that could be done within the Kontext framework. 

- **spatial co-influence of expression**. Analysing the correlation between $X_g$ and $C_{g^\prime}$ could possibly detect genes that are spatially repulsive to each other or that are always expressed together in some regions. 
- **Niche to expression**. Same idea. Correlation between the expression of a gene and the presence of a cell type in the local microenvironment could lead to unknown interactions. 
- **Post-segmentation denoising.** Transcripts are often assigned to the wrong cell during segmentation, a statistical model on $X$ and $C$ could be used for local denoising. 
- And so on. 

**Question.** 

Try to implement one of these ideas or invent your own additional analysis based on the Kontext framework. 


In [ ]:
# To fill 

# 5 - Another dataset 

Now you are ready to complete your own analysis. 

You can either load your own dataset if you have one or analyse one of these two : 
- Xenium Human breast cancer (https://www.nature.com/articles/s41467-023-43458-x)
- One section of MERFISH Mouse brain atlas (https://www.nature.com/articles/s41586-023-06808-9)



In [ ]:
# Load xenium human breast cancer 
# adata = sc.read_h5ad('./data/xenium_breast_cancer.h5ad')
adata = sc.read_h5ad('../data/xenium_breast_cancer.h5ad')
print(adata)
space_plot(adata,'annotation',s=5)


# Load MERFISH mouse brain section 
# adata = sc.read_h5ad('./data/mouse_brain_C57BL6J_1_053.h5ad')
adata = sc.read_h5ad('../data/mouse_brain_C57BL6J_1_053.h5ad')
print(adata)
space_plot(adata,'annotation',s=5)

**Question.** 

Use Kontext to perform your own analysis. 

In [ ]:
# To fill 

### 6 - Conclusion 

Congratulations, you managed to finish this workshop. 

The package and associated preprint should be available soon. 

Do not hesitate to send me your feedbacks on this notebook and on the Kontext project (anthony.ozier-lafontaine@pasteur.fr)

As I plan to have an academic carreer in omics data analysis and as a passionate of answering biological questions with mathematics, don't hesitate to contact me for future collaborations or brainstorming.  